In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# .env dosyası proje kökünde, notebook'tan yukarı çık
load_dotenv("../.env")
engine = create_engine(os.getenv("SQLALCHEMY_DATABASE_URL"))

print("Engine ready:", engine.url.host, engine.url.database)

Engine ready: localhost monetization_db


In [2]:
users = pd.read_sql("SELECT * FROM raw_user_source", engine)

print("Shape:", users.shape)
print("\nKolonlar:")
print(users.dtypes)
print("\nİlk 3 satır:")
users.head(3)

Shape: (15155, 7)

Kolonlar:
user_id                 object
install_date    datetime64[ns]
install_time            object
type                    object
version                 object
platform                object
country                 object
dtype: object

İlk 3 satır:


,user_id,install_date,install_time,type,version,platform,country
0,6E8E2F3F6A34809ECE127F006709FBEF,2024-08-01,03:17:32,Facebook Ads,2.6.25,android,US
1,D7CF81985E633F6AFA5FDA55D9EF68C2,2024-08-01,18:28:25,Facebook Ads,2.5.5,ios,US
2,D653987C55DC5DD8B30BAEADC5596315,2024-08-01,06:44:00,Facebook Ads,2.6.21,android,US


In [3]:
print("=== ACQUISITION KANALI ===")
print(users['type'].value_counts())
print(f"\nToplam install: {len(users):,}")

print("\n=== ÜLKE DAĞILIMI ===")
print(users['country'].value_counts().head(10))

print("\n=== PLATFORM ===")
print(users['platform'].value_counts())

=== ACQUISITION KANALI ===
type
organic         8797
Google Ads      2502
Facebook Ads    2479
Tik Tok Ads     1361
Name: count, dtype: int64

Toplam install: 15,155

=== ÜLKE DAĞILIMI ===
country
US    15155
Name: count, dtype: int64

=== PLATFORM ===
platform
android    7822
ios        7333
Name: count, dtype: int64


In [4]:
purchases = pd.read_sql("SELECT * FROM raw_purchases", engine)

print("Shape:", purchases.shape)
print("\nUnique paying users:", purchases['user_id'].nunique())
print(f"Conversion rate: {purchases['user_id'].nunique() / len(users) * 100:.2f}%")

print("\n=== HARCAMA İSTATİSTİKLERİ ===")
print(purchases['value_in_USD'].describe())

print("\n=== LTV İSTATİSTİKLERİ ===")
print(purchases['ltv'].describe())

Shape: (1427, 5)

Unique paying users: 1378
Conversion rate: 9.09%

=== HARCAMA İSTATİSTİKLERİ ===
count    1.427000e+03
mean     1.300000e+00
std      2.221224e-16
min      1.300000e+00
25%      1.300000e+00
50%      1.300000e+00
75%      1.300000e+00
max      1.300000e+00
Name: value_in_USD, dtype: float64

=== LTV İSTATİSTİKLERİ ===
count    1427.000000
mean        3.767008
std         2.502252
min         1.160000
25%         1.160000
50%         3.250000
75%         5.750000
max        42.000000
Name: ltv, dtype: float64


In [13]:
ads = pd.read_sql("SELECT * FROM raw_ad_spend", engine)

# Spend by channel (ad_spend tarafından)
spend_by_channel = ads.groupby('ad_platform')['ad_spend'].sum()

# Her user için max LTV (cumulative)
user_ltv = purchases.groupby('user_id')['ltv'].max().reset_index()

# CLEAN kanal kullan
revenue = user_ltv.merge(
    users[['user_id', 'type_clean']],
    on='user_id',
    how='left',
)
revenue_by_channel = revenue.groupby('type_clean')['ltv'].sum()
installs_by_channel = users['type_clean'].value_counts()

# Tüm kanalları topla
all_channels = sorted(set(installs_by_channel.index) | set(spend_by_channel.index))

print(f"{'Channel':<20} {'Installs':>9} {'Spend':>10} {'Revenue':>10} {'CPI':>8} {'ROAS':>8}")
print("-" * 75)

for channel in all_channels:
    installs = installs_by_channel.get(channel, 0)
    spend = spend_by_channel.get(channel, 0)
    rev = revenue_by_channel.get(channel, 0)
    cpi = spend / installs if installs > 0 else 0
    roas = rev / spend if spend > 0 else None

    cpi_str = f"${cpi:.2f}" if installs > 0 else "—"
    if roas is None:
        roas_str = "∞" if rev > 0 else "—"
        verdict = "organic" if channel == "Organic" else "no spend"
    else:
        roas_str = f"{roas:.2f}"
        verdict = "✓ kârlı" if roas > 1 else ("≈ borderline" if roas > 0.7 else "✗ zarar")

    print(f"{channel:<20} {installs:>9,} ${spend:>9,.0f} ${rev:>9,.0f} {cpi_str:>8} {roas_str:>8}  {verdict}")

Channel               Installs      Spend    Revenue      CPI     ROAS
---------------------------------------------------------------------------
Facebook Ads             2,479 $    8,400 $      798    $3.39     0.10  ✗ zarar
Google Ads               2,502 $    7,600 $      910    $3.04     0.12  ✗ zarar
Organic                  8,797 $        0 $    2,934    $0.00        ∞  organic
TikTok Ads               1,361 $    4,000 $      627    $2.94     0.16  ✗ zarar


In [8]:
print("=== user_source.type (unique) ===")
print(users['type'].value_counts())
print()
print("=== ad_spend.ad_platform (unique) ===")
print(ads['ad_platform'].value_counts())

=== user_source.type (unique) ===
type
organic         8797
Google Ads      2502
Facebook Ads    2479
Tik Tok Ads     1361
Name: count, dtype: int64

=== ad_spend.ad_platform (unique) ===
ad_platform
Facebook Ads    24
Google Ads      20
TikTok Ads      16
Name: count, dtype: int64


In [10]:
print("Ad spend toplam satır:", len(ads))
print("\nİlk 5 satır:")
print(ads.head())
print("\nToplam spend platform başına:")
print(ads.groupby('ad_platform')['ad_spend'].sum())
print("\nToplam spend:", ads['ad_spend'].sum())

Ad spend toplam satır: 60

İlk 5 satır:
  campaign_id           campaign_name   ad_platform       date  ad_spend  \
0  CAMPIOS001  Summer Splash Discount  Facebook Ads 2024-08-01       500   
1  CAMPAND001  Summer Splash Discount  Facebook Ads 2024-08-01       500   
2  CAMPIOS002  Summer Gaming Festival    Google Ads 2024-08-02       300   
3  CAMPAND002  Summer Gaming Festival    Google Ads 2024-08-02       300   
4  CAMPIOS003   Epic Game Launch 2025    TikTok Ads 2024-08-03       200   

  geo_location platform  
0           US      ios  
1           US  android  
2           US      ios  
3           US  android  
4           US      ios  

Toplam spend platform başına:
ad_platform
Facebook Ads    8400
Google Ads      7600
TikTok Ads      4000
Name: ad_spend, dtype: int64

Toplam spend: 20000


In [12]:
# Naming normalize
type_normalize = {
    'organic': 'Organic',
    'Tik Tok Ads': 'TikTok Ads',
    'Facebook Ads': 'Facebook Ads',
    'Google Ads': 'Google Ads',
}
users['type_clean'] = users['type'].map(type_normalize).fillna(users['type'])

print(users['type_clean'].value_counts())

type_clean
Organic         8797
Google Ads      2502
Facebook Ads    2479
TikTok Ads      1361
Name: count, dtype: int64
